# Personal Project Summer 2026
By Evan O'Malley

Context of the project  
- Recreational Project For Resumè Building
- Dataset from Pew Research

Goals of the project  
- Focus on efficient and effective use of library functions
- Focus on making legible work for potential presentation
- Focus on building a robust resource library for the notebook

## Parent Initialization

### Notebook Setup  
Imports files and packages

In [1]:
# Necessary Imports
# Pretty Lean
import pandas as pd, numpy as np
from warnings import simplefilter
simplefilter(action="ignore", category=pd.errors.PerformanceWarning)

rel_path = "Data/Western_Europe_Public_Data_Church_Tax_Added.csv"
codebook_rel_path = "Codebooks/codebook.txt"

parent_df = pd.read_csv(rel_path, skipinitialspace=True)
with open(codebook_rel_path) as f:
    codebook = f.read()

### Precursor Structures
Paragraph

In [2]:
class OrdLeg:
    def __init__(self, order: str, legend: dict):
        self.order = order
        self.legend = legend
        
    def ord(self):
        return self.order
    
    def leg(self):
        return self.legend
    
    def values(self):
        return self.legend.values()

    def keys(self):
        return self.legend.keys()

    def items(self):
        return self.legend.items()

    def __repr__(self):
        return f"{self.order}, {self.legend}"

    def __str__(self):
        return f"{self.order}, {self.legend}"

    def __getitem__(self, i):
        return self.legend[i]

### File Parsing
Makes codebook.txt into usable object

In [3]:
def parse_next(terminator):
    assert len(terminator) == 1, "Terminator must be one character"
    global s
    global codebook
    
    passage = ""
    while codebook[s] != terminator:
        passage += codebook[s]
        s += 1
    s += 1
        
    return passage


s = 0
Legends = {}

while s < len(codebook):
    col_legend = {}
    name = parse_next(':')
    order = parse_next('{')
    s += 1
    
    if "[COUNTRY]" in name:
        while codebook[s] != '!':
            i = int(parse_next(' '))
            val = parse_next('\n')
            col_legend[i] = val
        s += 1

        Legends[name.replace("[COUNTRY]", "")] = OrdLeg(order, {})
        
        while codebook[s] != '}':
            c = codebook[s:s+3]
            s += 5
            
            while codebook[s] != '}':
                i = int(parse_next(' '))
                val = parse_next('\n')
                col_legend[i] = val
            s += 2

            Legends[name.replace("[COUNTRY]", c)] = OrdLeg(order, col_legend)
        s += 2
            
    else:
        while codebook[s] != '}':
            i = int(parse_next(' '))
            val = parse_next('\n')
            col_legend[i] = val
        s += 2
        
        Legends[name] = OrdLeg(order, col_legend)

### Restitching
These cells altar the parent DataFrame and cannot be rerun

In [4]:
# Move QRID column to index
parent_df.index = parent_df["QRID"]
parent_df = parent_df.drop("QRID", axis = 1)

In [5]:
# Fix respose layout for Q9
Q9_cols = parent_df.columns.map(lambda x: x[1] == "9")
Q9_cols = parent_df.iloc[:, Q9_cols]

def get_Q9(row: pd.Series):
    i = row[row == 1].index
    if len(i) == 0:
        return np.nan
        
    else:
        return int(i[0][3])

parent_df.insert(16, "Q9", Q9_cols.apply(get_Q9, axis = 1))
parent_df = parent_df.drop(Q9_cols, axis = 1)

In [6]:
# Removing QS1... variables becuase they stand for regions and are indecipherable or redacted
# Removing qbornmoverec variable because values are incomprehensible or redacted
QS1_cols = [i for i in parent_df.columns if i.lower()[:3] == "qs1"]
parent_df = parent_df.drop(QS1_cols, axis = 1)
parent_df = parent_df.drop("qbornmoverec", axis = 1)

In [7]:
# Knit QIDEOLOGY, QIDEOLOGYa, and QIDEOLOGYb,
# as they are nearly identical
# FIXME THIS NEEDS TO BE NORMALIZED TO 1-6
parent_df["QIDEOLOGY"] = parent_df["QIDEOLOGY"].fillna(parent_df["QIDEOLOGYa"]).fillna(parent_df["QIDEOLOGYb"])
parent_df = parent_df.drop(["QIDEOLOGYa", "QIDEOLOGYb"], axis = 1)

In [8]:
# Repair column naming scheme
def title_scheme(title: str):
    # Cumulatively adjusts column titles according to scheme described below
    new_title = title

    # Remove instances of "rec", signifying recoded variables

    if new_title[-3:].lower() == "rec":
        new_title = new_title[:-3]

    # Normalize Capitalization Scheme
    # "country" -> "Country"
    # "QCURREL", "qcurrel" -> "QCurrel"
    
    if new_title[0].lower() != 'q':
        new_title = new_title.title()
        
    else:
        new_title = 'Q' + new_title[1:].title()
    
    # Capitalize suffixes signifying country
    # "QDenomaut" -> "QDenomAUT"
    
    if new_title[-3:].upper() in Legends["Country"].values():
        new_title = new_title[:-3] + new_title[-3:].upper()
    if new_title[-4:-1].upper() in Legends["Country"].values():
        new_title = new_title[:-4] + new_title[-4:-1].upper() + new_title[-1]

    # Some questions are divided into cases a, b, c, etcetera
    # Denotation for this will be separated from the main title and uncapitalized
    # "Q4A", "Q4B" -> "Q4_a", Q4_b"
    # These are dicipherable by last 2 characters of the title

    NumCap = new_title[-1].isupper() and new_title[-2] in "0123456789"
    CapUncap = new_title[-1].islower() and new_title[-2].isupper()
    if NumCap or CapUncap:
        new_title = new_title[:-1] + "_" + new_title[-1].lower()

    # Choice adjustments
    if new_title[:4] == "QPty":
        if new_title[4] == 'a':
            new_title = new_title[:4] + "potvot" + new_title[5:]
        elif new_title[4] == 'b':
            new_title = new_title[:4] + "fvr" + new_title[5:]
        else:
            new_title = new_title[:4] + "cls" + new_title[4:]

    rename_key = {"QCitizen1" : "QCitizen",
                  "QBornc" : "QBornmthr",
                  "QBorne" : "QBornfthr",
                  "QChilda" : "QChild",
                  "QHhch" : "QParent"}

    if new_title in rename_key.keys():
        new_title = rename_key[new_title]
    
    return new_title

parent_df = parent_df.rename(title_scheme, axis = 1)

### Further Preparations
Functions for table manipulations

In [9]:
# Cardinality function for titles
# Only works on columns post-rename
def cardinality(title: str):
    if title in Legends:
        return Legends[title].ord()
        
    if title[-2] == '_':
        return Legends[title[:-1]].ord()

    raise NotImplementedError(f"Column {title} inappropriately named")

In [10]:
# Function to divide tables into cardinalities
def Cardinate(df: pd.DataFrame, cardinality_arg: str):
    return df.loc[:, df.apply(lambda x: cardinality(x.name)) == cardinality_arg]

In [11]:
# Create Generalized list of Qs in order
# This will be useful later
# This has not been useful yet
def strip_countries(title: str):
    # Code primary lifted from title_scheme function
    if title[-3:] in Legends["Country"].values():
        return title[:-3]
    if title[-5:-2] in Legends["Country"].values():
        return title[:-5] + title[-2:]

    return title
        
survey_order = parent_df.apply(lambda x: strip_countries(x.name)).drop_duplicates().__array__()

In [12]:
# Function to disambiguate and ambiguate enumerations
def disambiguate(col: pd.Series):
    if col.dtypes == int:
        return col.map(lambda x: Legends[col.name][x])
    else:
        return col

def ambiguate(col: pd.Series):
    if col.dtypes == int:
        return col
    else:
        reverse_legend = {y:x for x,y in Legends[col.name].items()}
        return col.map(lambda x: reverse_legend[x])

In [13]:
# Function to divide tables into countries
# Returns a dictionary of the division
def Nationate(df: pd.DataFrame):
    country_dfs = {}
    if "Country" not in df:
        df = df.join(parent_df["Country"], how = "left")
    df["Country"] = disambiguate(df["Country"])

    for c in df["Country"].unique():
        country_df = df.copy()
        country_df = country_df.loc[country_df["Country"] == c]
        country_df = country_df.loc[:, country_df.apply(lambda x: any(x.notna()))]
        country_df = country_df.rename(strip_countries, axis = 1)
        country_dfs[c] = country_df.drop("Country", axis = 1)
        
    return country_dfs

### Summary of Parent Initialization
Coagulate essenstial information  
- Legends object  
- OrdLeg class and methods

Prepare parent dataframe  
- Column cropping  
- Rename scheme

Create DataFrame editing tools  
- Cardinate  
- Nationate  
- Supplementary functions

## Rudamentary Analysis

### Poking Around
Start inventing some summary statistics

In [14]:
parent_df

,Country,Weight,Q1,Q2,Q4_a,Q4_b,Q4_c,Q4_d,Q4_e,Q4_f,Q4_g,Q4_h,Q5,Q6,Q7,Q8,Q9,Q10_a,Q10_b,Q10_c,Q11,Q13,Q14_a,Q14_b,QCurrel,QCurreld,QDenomAUT,QDenomBEG,QDenomDNK,QDenomFIN,QDenomFRA,QDenomDEU,QDenomIRL,QDenomITA,QDenomNLD,QDenomNOR,QDenomPRT,QDenomSLO,QDenomESP,QDenomSWE,...,QIncjGBR,QInckGBR,QIdeology,QPtypotvotAUT,QPtypotvotBEG,QPtypotvotDNK,QPtypotvotFIN,QPtypotvotFRA,QPtypotvotDEU,QPtypotvotIRL,QPtypotvotITA,QPtypotvotNLD,QPtypotvotNOR,QPtypotvotPRT,QPtypotvotSLO,QPtypotvotESP,QPtypotvotSWE,QPtypotvotCHE,QPtypotvotGBR,QPtyclsAUT,QPtyclsBEG,QPtyclsDNK,QPtyclsFIN,QPtyclsFRA,QPtyclsDEU,QPtyclsIRL,QPtyclsITA,QPtyclsNLD,QPtyclsNOR,QPtyclsPRT,QPtyclsSLO,QPtyclsESP,QPtyclsSWE,QPtyclsCHE,QPtyclsGBR,QCitizen,QBorn,QBornmthr,QBornfthr,Isced
QRID,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1006015,1,0.312580,1,2,2,2,2,1,2,1,2,2,2,2,2,3,NaN,1,1,2,1,1.0,4.0,NaN,1,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,2.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,1,1,5
1012247,1,2.165165,1,2,2,2,2,1,2,2,2,2,2,2,1,2,NaN,3,2,3,3,2.0,NaN,1.0,92,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,4.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,1,1,3
1012251,1,1.583167,1,2,1,2,2,2,2,2,2,2,2,3,2,2,NaN,3,3,4,2,1.0,4.0,NaN,91,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,2.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,1,1,3
1012271,1,0.705469,1,2,1,2,2,2,2,2,2,2,2,2,1,1,7.0,3,1,3,1,1.0,2.0,NaN,1,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,4.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,1,1,5
1012279,1,2.420254,1,2,2,2,2,2,2,2,2,2,2,2,1,2,NaN,3,3,3,2,1.0,4.0,NaN,1,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,1,1,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13646329,13,0.636224,2,2,2,2,2,2,2,2,2,2,2,2,1,2,NaN,4,2,4,1,2.0,NaN,1.0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,NaN,...,NaN,NaN,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,1,97,97,1,3
13646341,13,0.235802,2,1,2,1,2,2,2,2,2,2,2,2,1,2,NaN,2,2,4,1,1.0,3.0,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,NaN,...,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,NaN,NaN,NaN,1,1,1,1,3
13685713,13,0.249925,1,2,2,2,2,2,2,2,2,2,1,2,1,2,NaN,3,2,3,2,2.0,NaN,2.0,92,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,98.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,NaN,NaN,NaN,1,1,1,1,5
